## HW Assignment 2

In this assignment, we will learn how to use Apache Spark DataFrames and the MLLib package used for machine learning.
Run the code below to start up a local Spark instance.

### Assignment submission group
- Group member 1: Chakat Srisuvanutna (cs635@drexel.edu)
- Group member 2: Name (email)

!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://www-eu.apache.org/dist/spark/spark-2.4.5/spark-2.4.5-bin-hadoop2.7.tgz
!tar xf spark-2.4.5-bin-hadoop2.7.tgz

In [15]:
# import os
# os.environ["JAVA_HOME"]='/usr/local/opt/openjdk@8'
# os.environ["SPARK_HOME"] = "/content/spark-2.4.5-bin-hadoop2.7"
import subprocess
import os

result = subprocess.run(['java', '-XshowSettings:property', '-version'], 
                       capture_output=True, text=True)

for line in result.stderr.split('\n'):
    if 'java.home' in line:
        java_home = line.split('=')[1].strip()
        os.environ["JAVA_HOME"] = java_home
        print(f"JAVA_HOME = {java_home}")

JAVA_HOME = C:\Program Files\Java\jdk-17


In [2]:
import pyspark
import os

spark_home = os.path.dirname(pyspark.__file__)
os.environ["SPARK_HOME"] = spark_home
print(f"SPARK_HOME = {spark_home}")

SPARK_HOME = C:\Users\icejb\anaconda3\Lib\site-packages\pyspark


In [3]:
!pip install -q findspark
!pip install pyspark

In [4]:
# from google.colab import drive
# drive.mount('/content/gdrive')

FILE_PATH = 'travel insurance.csv' 

if os.path.exists(FILE_PATH):
    print(f"✅ Found: {FILE_PATH}")
else:
    print(f"❌ Not found: {FILE_PATH}")

✅ Found: travel insurance.csv


In [5]:
from pyspark.sql import SparkSession
APP_NAME = "HW2"
spark = SparkSession.builder.appName(APP_NAME).getOrCreate()
spark

1. a. In the first part of this assignment, we will load a dataset and discover some insight about the data.
Load the travel dataset provided in the assignment with the option inferSchema set to true. Print 20 rows from the table

In [7]:
df = spark.read.csv(FILE_PATH, header=True, inferSchema=True)
df.show(20)

+------+-------------+--------------------+--------------------+-----+--------+--------------------+---------+--------------------+------+---+
|Agency|  Agency Type|Distribution Channel|        Product Name|Claim|Duration|         Destination|Net Sales|Commision (in value)|Gender|Age|
+------+-------------+--------------------+--------------------+-----+--------+--------------------+---------+--------------------+------+---+
|   CBH|Travel Agency|             Offline|  Comprehensive Plan|   No|     186|            MALAYSIA|    -29.0|                9.57|     F| 81|
|   CBH|Travel Agency|             Offline|  Comprehensive Plan|   No|     186|            MALAYSIA|    -29.0|                9.57|     F| 71|
|   CWT|Travel Agency|              Online|Rental Vehicle Ex...|   No|      65|           AUSTRALIA|    -49.5|                29.7|  NULL| 32|
|   CWT|Travel Agency|              Online|Rental Vehicle Ex...|   No|      60|           AUSTRALIA|    -39.6|               23.76|  NULL| 32|

b. Rename the Commision (in value) column to Commission. Assign this dataframe to a new variable.

In [8]:
df2 = df.withColumnRenamed("Commision (in value)", "Commission")
df2.printSchema()

root
 |-- Agency: string (nullable = true)
 |-- Agency Type: string (nullable = true)
 |-- Distribution Channel: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Claim: string (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- Destination: string (nullable = true)
 |-- Net Sales: double (nullable = true)
 |-- Commission: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)



c. Compute the count of policies for each destination. Print the top 10 destinations by the count of policies.

In [26]:
from pyspark.sql.functions import count, desc

dest_count = (
    df2
    .groupBy("Destination")
    .agg(count("*").alias("Policy Count"))
    .orderBy(desc("Policy Count"))
)
dest_count.show(10)

+-------------+------------+
|  Destination|Policy Count|
+-------------+------------+
|    SINGAPORE|       13255|
|     MALAYSIA|        5930|
|     THAILAND|        5894|
|        CHINA|        4796|
|    AUSTRALIA|        3694|
|    INDONESIA|        3452|
|UNITED STATES|        2530|
|  PHILIPPINES|        2490|
|    HONG KONG|        2411|
|        INDIA|        2251|
+-------------+------------+
only showing top 10 rows


d. What is the mean age for customers who filed a claim? What is the mean age for customers who did not file a claim?
Print the mean age for both customers who filed and those who didn't file.

In [27]:
from pyspark.sql.functions import mean

df2.groupBy("Claim").agg(mean("Age").alias("Mean Age")).show()

+-----+------------------+
|Claim|          Mean Age|
+-----+------------------+
|   No|39.989823554864664|
|  Yes| 38.63430420711974|
+-----+------------------+



e. Which travel agency made the most amount of money in commission? Compute the total amount of commission for each agency and print the top 10 agencies ordered by the amount.

In [28]:
from pyspark.sql.functions import sum as spark_sum, desc

agency_comm = (
    df2
    .groupBy("Agency")
    .agg(spark_sum("Commission").alias("Total Commission"))
    .orderBy(desc("Total Commission"))
)
agency_comm.show(10)

+------+------------------+
|Agency|  Total Commission|
+------+------------------+
|   CWT|  277825.680000016|
|   C2B|169747.34000000294|
|   JZI|  74471.2499999991|
|   LWC| 51169.12999999992|
|   JWT|16208.399999999958|
|   KML|  8550.38000000001|
|   TST|           5556.25|
|   RAB| 5239.199999999987|
|   ART|           3493.35|
|   ADM|3136.8999999999996|
+------+------------------+
only showing top 10 rows


2. a. The second part of the assignment will be to create a transformation pipeline. Remove all rows with missing data, convert all strings to integers, create dummy variables and assemble a feature vector. Use only the following variables as predictors: Agency Type, Distribution Channel, Duration, Net Sales, Commission, Gender, Age. Use Claim as the response variable.

In [10]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

selected_cols = ["Agency Type", "Distribution Channel", "Duration",
                 "Net Sales", "Commission", "Gender", "Age", "Claim"]
df_clean = df2.select(selected_cols).na.drop()

# StringIndexer — strings → integers
indexers = [
    StringIndexer(inputCol="Agency Type",          outputCol="Agency Type Idx"),
    StringIndexer(inputCol="Distribution Channel", outputCol="Dist Channel Idx"),
    StringIndexer(inputCol="Gender",               outputCol="Gender Idx"),
    StringIndexer(inputCol="Claim",                outputCol="label"),
]

# OneHotEncoder — integers → dummy variables
encoder = OneHotEncoder(
    inputCols= ["Agency Type Idx", "Dist Channel Idx", "Gender Idx"],
    outputCols=["Agency Type Vec", "Dist Channel Vec", "Gender Vec"]
)

# VectorAssembler — combine all features
assembler = VectorAssembler(
    inputCols=["Agency Type Vec", "Dist Channel Vec", "Gender Vec",
               "Duration", "Net Sales", "Commission", "Age"],
    outputCol="features"
)

# Build & run pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler])
pipeline_model = pipeline.fit(df_clean)
df_transformed = pipeline_model.transform(df_clean)

df_transformed.select("features", "label").show(5)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[0.0,0.0,0.0,186....|  0.0|
|[0.0,0.0,0.0,186....|  0.0|
|[1.0,1.0,0.0,66.0...|  0.0|
|[1.0,1.0,1.0,1.0,...|  0.0|
|[0.0,1.0,0.0,53.0...|  0.0|
+--------------------+-----+
only showing top 5 rows


b. Split the data into train and test with 20% of the data in the test sample.

In [11]:
train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)

print(f"Train size: {train_df.count()}")
print(f"Test size:  {test_df.count()}")

Train size: 14648
Test size:  3571


c. Apply a logistic regression model to predict the probaility that a customer will make a claim. Use the training data to produce a model and then test it using the test dataset.

In [12]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_df)

train_predictions = lr_model.transform(train_df)
test_predictions  = lr_model.transform(test_df)

test_predictions.select("label", "prediction", "probability").show(5)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       0.0|[0.99477991854210...|
|  0.0|       0.0|[0.98483490715427...|
|  0.0|       0.0|[0.98052833414737...|
|  0.0|       0.0|[0.96733746946604...|
|  0.0|       0.0|[0.97435785164729...|
+-----+----------+--------------------+
only showing top 5 rows


d. Compute the accuracy for the train and test datasets.

In [13]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

train_acc = evaluator.evaluate(train_predictions)
test_acc  = evaluator.evaluate(test_predictions)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

Train Accuracy: 0.9653
Test Accuracy:  0.9622
